In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
print("\n📂 Loading raw data...")
df = pd.read_csv('./archive/dataset/dataset/Annual_P_L_1_final.csv')
print(f"   Raw data: {len(df)} rows")


📂 Loading raw data...
   Raw data: 4668 rows


In [3]:
print("\n🧹 Preprocessing...")

# Remove extreme outliers
df_clean = df[(df['OPM'] > -100) & (df['OPM'] < 200)].copy()
print(f"   After outlier removal: {len(df_clean)} rows")

# Winsorization
lower = df_clean['OPM'].quantile(0.01)
upper = df_clean['OPM'].quantile(0.99)
df_clean['OPM_clean'] = df_clean['OPM'].clip(lower, upper)

# Business rule caps
df_clean['OPM_clean'] = df_clean['OPM_clean'].clip(-50, 100)

# Create target categories
df_clean['opm_category'] = pd.cut(
    df_clean['OPM_clean'],
    bins=[-np.inf, 0, 10, 20, np.inf],
    labels=[0, 1, 2, 3]
)


🧹 Preprocessing...
   After outlier removal: 4167 rows


In [4]:
print("\n⚙️  Feature engineering...")

created_features = []

# 1. COST STRUCTURE RATIOS (4 features)
print("\n1️⃣ Cost Structure Ratios")

# Material cost ratio
df_clean['material_cost_ratio'] = (
    df_clean['Material cost last year'] / 
    df_clean['Sales last year'].replace(0, np.nan) * 100
)
lower_mat = df_clean['material_cost_ratio'].quantile(0.01)
upper_mat = df_clean['material_cost_ratio'].quantile(0.99)
df_clean['material_cost_ratio'] = df_clean['material_cost_ratio'].clip(
    lower_mat, upper_mat
).clip(-10, 150)
created_features.append('material_cost_ratio')
print(f"   ✅ material_cost_ratio")

# Employee cost ratio
df_clean['employee_cost_ratio'] = (
    df_clean['Employee cost last year'] / 
    df_clean['Sales last year'].replace(0, np.nan) * 100
)
lower_emp = df_clean['employee_cost_ratio'].quantile(0.01)
upper_emp = df_clean['employee_cost_ratio'].quantile(0.99)
df_clean['employee_cost_ratio'] = df_clean['employee_cost_ratio'].clip(
    lower_emp, upper_emp
).clip(-10, 150)
created_features.append('employee_cost_ratio')
print(f"   ✅ employee_cost_ratio")

# Total cost ratio
df_clean['total_cost_ratio'] = (
    df_clean['material_cost_ratio'] + df_clean['employee_cost_ratio']
)
created_features.append('total_cost_ratio')
print(f"   ✅ total_cost_ratio")

# Depreciation ratio
df_clean['depreciation_ratio'] = (
    df_clean['Depreciation'] / 
    df_clean['Sales'].replace(0, np.nan) * 100
)
lower_dep = df_clean['depreciation_ratio'].quantile(0.01)
upper_dep = df_clean['depreciation_ratio'].quantile(0.99)
df_clean['depreciation_ratio'] = df_clean['depreciation_ratio'].clip(
    lower_dep, upper_dep
).clip(-5, 50)
created_features.append('depreciation_ratio')
print(f"   ✅ depreciation_ratio")

# 2. GROWTH & MOMENTUM (3 features)
print("\n2️⃣ Growth & Momentum")

# Sales growth
df_clean['sales_growth'] = (
    (df_clean['Sales'] - df_clean['Sales last year']) / 
    df_clean['Sales last year'].replace(0, np.nan) * 100
)
df_clean['sales_growth'] = df_clean['sales_growth'].clip(
    df_clean['sales_growth'].quantile(0.01),
    df_clean['sales_growth'].quantile(0.99)
).clip(-90, 500)
created_features.append('sales_growth')
print(f"   ✅ sales_growth")

# OPM change
df_clean['opm_change'] = (
    df_clean['OPM_clean'] - df_clean['OPM last year']
)
df_clean['opm_change'] = df_clean['opm_change'].clip(
    df_clean['opm_change'].quantile(0.01),
    df_clean['opm_change'].quantile(0.99)
)
created_features.append('opm_change')
print(f"   ✅ opm_change")

# Improving OPM (binary)
df_clean['improving_opm'] = (df_clean['opm_change'] > 0).astype(int)
created_features.append('improving_opm')
print(f"   ✅ improving_opm")

# 3. COMPANY SIZE (2 features)
print("\n3️⃣ Company Size")

# Log sales
df_clean['log_sales'] = np.log1p(df_clean['Sales'].clip(lower=0))
created_features.append('log_sales')
print(f"   ✅ log_sales")

# Log market cap
df_clean['log_market_cap'] = np.log1p(
    df_clean['Market Capitalization'].clip(lower=0)
)
created_features.append('log_market_cap')
print(f"   ✅ log_market_cap")

# 4. BINARY INDICATORS (2 features)
print("\n4️⃣ Binary Indicators")

# Profitable last year
df_clean['profitable_last_year'] = (
    df_clean['Profit after tax last year'] > 0
).astype(int)
created_features.append('profitable_last_year')
print(f"   ✅ profitable_last_year")

# Positive OPM last year
df_clean['positive_opm_last_year'] = (
    df_clean['OPM last year'] > 0
).astype(int)
created_features.append('positive_opm_last_year')
print(f"   ✅ positive_opm_last_year")

# 5. EFFICIENCY (1 feature)
print("\n5️⃣ Efficiency Metrics")

# Asset turnover
df_clean['asset_turnover'] = (
    df_clean['Sales'] / 
    df_clean['Market Capitalization'].replace(0, np.nan)
)
df_clean['asset_turnover'] = df_clean['asset_turnover'].clip(
    df_clean['asset_turnover'].quantile(0.01),
    df_clean['asset_turnover'].quantile(0.99)
)
created_features.append('asset_turnover')
print(f"   ✅ asset_turnover")

print(f"\n   Total created features: {len(created_features)}")


⚙️  Feature engineering...

1️⃣ Cost Structure Ratios
   ✅ material_cost_ratio
   ✅ employee_cost_ratio
   ✅ total_cost_ratio
   ✅ depreciation_ratio

2️⃣ Growth & Momentum
   ✅ sales_growth
   ✅ opm_change
   ✅ improving_opm

3️⃣ Company Size
   ✅ log_sales
   ✅ log_market_cap

4️⃣ Binary Indicators
   ✅ profitable_last_year
   ✅ positive_opm_last_year

5️⃣ Efficiency Metrics
   ✅ asset_turnover

   Total created features: 12


In [5]:
print("\n🎯 Selecting features...")

# These are the EXACT features used in your correlation analysis
features_to_use = [
    #'Profit after tax last year',
    #'Sales last year',
    #'positive_opm_last_year',  # To zostaw - to binary flag
    #'profitable_last_year',
    #'material_cost_ratio',
    #'employee_cost_ratio',
    #'total_cost_ratio',
    #'log_market_cap',
    #'depreciation_ratio',
    #'asset_turnover',
    #'log_sales',

    'Profit after tax last year',
    'Sales last year',
    'positive_opm_last_year',
    'profitable_last_year',
    
    # Cost structure
    'material_cost_ratio',
    'employee_cost_ratio',
    'total_cost_ratio',
    'depreciation_ratio',
    
    # Growth
    #'sales_growth',
    #'opm_change',
    # 'improving_opm',
    
    # Size
    'log_sales',
    'log_market_cap',
    
    # Efficiency
    #'Return on capital employed',
    'asset_turnover',
    
    # Financial
    #'Interest',
    #'Depreciation',

]

print(f"   Features selected: {len(features_to_use)}")

# Remove rows with NaN in target
df_final = df_clean.dropna(subset=['opm_category'])
print(f"   After removing NaN in target: {len(df_final)} rows")

# Remove rows with NaN in features
df_final = df_final.dropna(subset=features_to_use)
print(f"   After removing NaN in features: {len(df_final)} rows")


🎯 Selecting features...
   Features selected: 11
   After removing NaN in target: 4167 rows
   After removing NaN in features: 4167 rows


In [6]:
print("\n📊 Preparing X and y...")

X = df_final[features_to_use].copy()
y = df_final['opm_category'].copy()

print(f"   X shape: {X.shape}")
print(f"   y shape: {y.shape}")
print(f"\n   Class distribution:")
for label in [0, 1, 2, 3]:
    count = (y == label).sum()
    pct = count / len(y) * 100
    print(f"      Class {label}: {count:4d} ({pct:5.1f}%)")


📊 Preparing X and y...
   X shape: (4167, 11)
   y shape: (4167,)

   Class distribution:
      Class 0:  587 ( 14.1%)
      Class 1: 1410 ( 33.8%)
      Class 2: 1135 ( 27.2%)
      Class 3: 1035 ( 24.8%)


In [7]:
print("\n✂️  Splitting data (80/20)...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y  # Keep class proportions
)

print(f"   Train: {len(X_train)} samples")
print(f"   Test:  {len(X_test)} samples")


✂️  Splitting data (80/20)...
   Train: 3333 samples
   Test:  834 samples


In [8]:
print("\n💾 Exporting files...")

import os
os.makedirs('data_ml', exist_ok=True)

X_train.to_csv('data_ml/X_train.csv', index=False)
X_test.to_csv('data_ml/X_test.csv', index=False)
y_train.to_csv('data_ml/y_train.csv', index=False, header=True)
y_test.to_csv('data_ml/y_test.csv', index=False, header=True)

# Save feature names
with open('data_ml/feature_names.txt', 'w') as f:
    f.write("Features used for ML:\n")
    f.write("="*50 + "\n")
    for i, feat in enumerate(features_to_use, 1):
        f.write(f"{i:2d}. {feat}\n")

print("   ✅ X_train.csv")
print("   ✅ X_test.csv")
print("   ✅ y_train.csv")
print("   ✅ y_test.csv")
print("   ✅ feature_names.txt")


💾 Exporting files...
   ✅ X_train.csv
   ✅ X_test.csv
   ✅ y_train.csv
   ✅ y_test.csv
   ✅ feature_names.txt


In [9]:
print("\n" + "="*80)
print("✅ DATA EXPORT COMPLETE!")
print("="*80)
print(f"\n📊 Summary:")
print(f"   Total samples:     {len(df_final)}")
print(f"   Training samples:  {len(X_train)}")
print(f"   Test samples:      {len(X_test)}")
print(f"   Number of features: {len(features_to_use)}")
print(f"   Number of classes:  4")
print(f"\n🚀 Ready for Random Forest training!")
print(f"   Run: python 02_train_random_forest.py")


✅ DATA EXPORT COMPLETE!

📊 Summary:
   Total samples:     4167
   Training samples:  3333
   Test samples:      834
   Number of features: 11
   Number of classes:  4

🚀 Ready for Random Forest training!
   Run: python 02_train_random_forest.py
